# Map Comparison Benchmark (binary-only)

Compares 4 sensors of varying noise level against a perfect-interpreter reference, using both full-raster pixel-level scoring (`src/map_comparison.py`) and the existing window-based sampling approaches (A–D). Config: `interpreter/experiments/configs/map_comparison_baseline.json`.

In [1]:
import sys, os, json
sys.path.insert(0, os.path.abspath('../../src'))

import numpy as np
import pandas as pd

from landscape_stack_collection import LandscapeStackCollection
from map_comparison import (
    perfect_reference_mask,
    pixel_level_metrics,
    pixel_level_metrics_for_collection,
    map_comparison_meta_analysis,
)

In [2]:
PROJ_ROOT = os.path.abspath('../../')

with open('../experiments/configs/map_comparison_baseline.json') as f:
    cfg = json.load(f)

stack_cfg = cfg['landscape_stack_config']
train_exp_cfg = dict(cfg['training_experiment_config'])
val_exp_cfg = dict(cfg['validation_experiment_config'])
train_exp_cfg['patch_dir'] = os.path.join(PROJ_ROOT, train_exp_cfg['patch_dir'])
val_exp_cfg['patch_dir'] = os.path.join(PROJ_ROOT, val_exp_cfg['patch_dir'])

sensor_names = [s['name'] for s in stack_cfg['sensors']]
sensor_names

['Sensor_Clean', 'Sensor_Low', 'Sensor_Medium', 'Sensor_High']

## 1. Training collection + binary classifiers per sensor

In [3]:
train_collection = LandscapeStackCollection.from_config(
    stack_template_cfg=stack_cfg,
    experiment_cfg=train_exp_cfg,
)

bc_cfg = cfg['binary_classifier_config']
train_collection.buildBinaryClassifier(
    sensor_names=bc_cfg['sensor_names'],
    n_strata=bc_cfg['n_strata'],
    samples_per_stratum=bc_cfg['samples_per_stratum'],
    sampling_seed=bc_cfg['sampling_seed'],
    fp_to_fn_ratio=bc_cfg['fp_to_fn_ratio'],
)
train_collection.applyBinaryClassifier(sensor_names=bc_cfg['sensor_names'])

/Users/rkennedy/Dropbox/caol/code/_claudecode/disturbance_uncertainty/src/disturbance_helper_functions.py:31: RuntimeWarning: overflow encountered in exp
  return 1.0 / (1.0 + np.exp(-k * (x - x0)))


## 2. Validation collection (20 scenes, disjoint from training)

In [4]:
used_tifs = [s.base_landscape.tif_path for s in train_collection.stacks]

val_collection = LandscapeStackCollection.from_config(
    stack_template_cfg=stack_cfg,
    experiment_cfg=val_exp_cfg,
    exclude_tif_paths=used_tifs,
)

assert set(used_tifs).isdisjoint({s.base_landscape.tif_path for s in val_collection.stacks}), \
    'validation set overlaps training set'

val_collection.loadBinaryModel(train_collection.binary_models)
val_collection.applyBinaryClassifier(sensor_names=bc_cfg['sensor_names'])

/Users/rkennedy/Dropbox/caol/code/_claudecode/disturbance_uncertainty/src/disturbance_helper_functions.py:31: RuntimeWarning: overflow encountered in exp
  return 1.0 / (1.0 + np.exp(-k * (x - x0)))


## 0. Validate the perfect-interpreter approximation

The extreme-slope detection/definition curves have no exact no-op (see config `notes`) — confirm empirically that the thresholded perfect-interpreter field agrees with ground truth almost everywhere before trusting it as a scoring reference.

In [5]:
for stack in val_collection.stacks[:3]:
    ref = perfect_reference_mask(stack)
    truth = stack.base_landscape.disturbed.astype(int)
    metrics = pixel_level_metrics(ref, truth)
    print(stack.stack_id, metrics)

DL_839e7ad0 {'agreement': np.float64(1.0), 'precision': np.float64(1.0), 'recall': np.float64(1.0), 'f1': np.float64(1.0), 'iou': np.float64(1.0), 'n_total': 48240}
DL_0ff4d232 {'agreement': np.float64(1.0), 'precision': np.float64(1.0), 'recall': np.float64(1.0), 'f1': np.float64(1.0), 'iou': np.float64(1.0), 'n_total': 45225}
DL_94dfc34b {'agreement': np.float64(1.0), 'precision': np.float64(1.0), 'recall': np.float64(1.0), 'f1': np.float64(1.0), 'iou': np.float64(1.0), 'n_total': 44844}


## 3. Pixel-level scoring (step 6)

In [6]:
pixel_df = pixel_level_metrics_for_collection(val_collection, sensor_names=bc_cfg['sensor_names'])
pixel_df.groupby('sensor_name')[['agreement', 'precision', 'recall', 'f1', 'iou']].mean()

,agreement,precision,recall,f1,iou
sensor_name,,,,,
Sensor_Clean,0.998576,0.934194,0.991164,0.959868,0.926382
Sensor_High,0.986308,0.710747,0.651051,0.674626,0.511447
Sensor_Low,0.992263,0.807338,0.855545,0.827247,0.709976
Sensor_Medium,0.989015,0.717874,0.826295,0.765121,0.622850


## 4. Window-based sampling (step 7, unmodified A–D)

In [7]:
ws_cfg = cfg['window_sample_config']

window_results = {}
for sensor_name in bc_cfg['sensor_names']:
    samples_a = val_collection.window_sample_A(sensor_name=sensor_name, **ws_cfg)
    confusion_a = val_collection.binary_confusion_from_samples(samples_a)
    olofsson_a = val_collection.olofsson_area_estimates(samples_a, sensor_name=sensor_name)
    window_results[sensor_name] = {
        'samples_A': samples_a,
        'confusion_A': confusion_a,
        'olofsson_A': olofsson_a,
    }

{name: r['confusion_A']['n_ij'] for name, r in window_results.items()}

{'Sensor_Clean': array([[17542,     9],
        [   13,   388]]),
 'Sensor_Low': array([[17468,    55],
        [   87,   342]]),
 'Sensor_Medium': array([[17402,    60],
        [  153,   337]]),
 'Sensor_High': array([[17410,   130],
        [  145,   267]])}

## 5. Meta-analysis (step 8): pixel-level vs. sampling-based metric deltas

In [8]:
meta_df = map_comparison_meta_analysis(
    val_collection,
    sensor_names=bc_cfg['sensor_names'],
    window_sample_fn=val_collection.window_sample_A,
    window_kwargs=ws_cfg,
)
meta_df

,sensor_a,sensor_b,metric,pixel_delta,sampling_delta
0,Sensor_Clean,Sensor_Low,agreement,0.006313,0.004792
1,Sensor_Clean,Sensor_Low,precision,0.126855,0.104303
2,Sensor_Clean,Sensor_Low,recall,0.135619,0.106218
3,Sensor_Clean,Sensor_Low,f1,0.132622,0.105421
4,Sensor_Clean,Sensor_Low,iou,0.216406,0.174770
5,Sensor_Clean,Sensor_Medium,agreement,0.009561,0.006854
6,Sensor_Clean,Sensor_Medium,precision,0.216320,0.171524
7,Sensor_Clean,Sensor_Medium,recall,0.164869,0.098446
8,Sensor_Clean,Sensor_Medium,f1,0.194747,0.140951
9,Sensor_Clean,Sensor_Medium,iou,0.303532,0.226684
